# Modelo Recomendador

El modelo recomendador se basa en la similitud de audio features mediante similitud coseno. Cuanto más cercanos son dos vectores de features, más parecidas son las canciones.

## 1. Importación de librerías y carga del dataset



In [1]:
# Librerías de manipulación de datos.
import pandas as pd 
import numpy as np 

# Librerías de machine learning.
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# Configuración de pandas.
pd.set_option('display.max_columns', None)

In [2]:
# Carga del dataset.
df_cluster = pd.read_csv('../data/processed/tracks_clustered.csv')
df = df_cluster.copy()
print(f'Dataset cargado: {df.shape[0]:,} filas')
df.head(3)

Dataset cargado: 113,422 filas


,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,duration_min,popularity_category,cluster,cluster_nombre
0,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.461,1,-6.746,0,0.1430,0.0322,0.000001,0.358,0.715,87.917,4,acoustic,3.84,Alta,3,Fiesta & Baile
1,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.166,1,-17.235,1,0.0763,0.9240,0.000006,0.101,0.267,77.489,4,acoustic,2.49,Alta,1,Acústico & Tranquilo
2,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.359,0,-9.734,1,0.0557,0.2100,0.000000,0.117,0.120,76.332,4,acoustic,3.51,Alta,1,Acústico & Tranquilo


## 2. Matriz de variables

In [3]:
# Selección de variables.
variables_modelo = ['danceability', 'energy', 'loudness', 'speechiness','acousticness', 'instrumentalness', 'liveness','valence', 'tempo']

# Escala de las variables.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[variables_modelo])

print(f'Matriz de variables: {X_scaled.shape}')
print(f'Variables usadas : {variables_modelo}')


Matriz de variables: (113422, 9)
Variables usadas : ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']


## 3. Función de recomendación

Basada en la similitud de canciones entre clusteres.

In [4]:
def recomendar_canciones(titulo, artista=None, n=10, df=df, X_scaled=X_scaled):
    """ 
    Recomienda canciones similares a sus variables dado el título de una canción.
    Args:
        titulo: Nombre de la canción (búsqueda parcial)
        artista: Filtrar por artista (opcional)
        n:  Número de recomendaciones
    
    Returns:
        DataFrame con las n canciones más similares
    """
    
    # Buscar la canción en el dataset.
    mask = df['track_name'].str.contains(titulo, case=False, na=False)
    if artista:
        mask &= df['artists'].str.contains(artista, case=False, na=False)
    candidatos = df[mask]
    
    if candidatos.empty:
        print(f'No se encontraron canciones que contengan "{titulo}"')
        return None
    
    # Si hay varias coincidencias, tomar la más popular.
    idx = candidatos['popularity'].idxmax()
    cancion_ref = df.loc[idx]
    
    print(f'Canción de referencia:')
    print(f"   {cancion_ref['track_name']} — {cancion_ref['artists']}")
    print(f'    Género: {cancion_ref["track_genre"]} | Cluster: {cancion_ref["cluster_nombre"]}')
    print(f'    Popularidad: {cancion_ref["popularity"]}')
    
    # Calcular similitud coseno entre la canción y todas las demás
    vector_ref = X_scaled[idx].reshape(1, -1)
    similitudes = cosine_similarity(vector_ref, X_scaled)[0]
    
    # Ordenar por similitud descendente, excluir la propia canción
    indices_similares = np.argsort(similitudes)[::-1]
    indices_similares = [i for i in indices_similares if i != idx][:n]
    
    recomendaciones = df.iloc[indices_similares][
        ['track_name', 'artists', 'track_genre', 'cluster_nombre', 'popularity',
         'danceability', 'energy', 'valence']
    ].copy()
    recomendaciones['similitud'] = similitudes[indices_similares].round(4)
    recomendaciones = recomendaciones.reset_index(drop=True)
    recomendaciones.index += 1
    
    return recomendaciones



In [5]:
# Primera prueba
titulo = "Blinding Lights"
artista = "The Weeknd"
n = 10
recomendaciones = recomendar_canciones(titulo, artista, n)
recomendaciones

Canción de referencia:
   Blinding Lights — The Weeknd
    Género: pop | Cluster: Rock & Intenso
    Popularidad: 91


,track_name,artists,track_genre,cluster_nombre,popularity,danceability,energy,valence,similitud
1,平凡人的自傳 - Rap Version,ONE PROMISE,cantopop,Rock & Intenso,22,0.507,0.717,0.353,0.9916
2,Viah,Jass Manak,hip-hop,Rock & Intenso,59,0.533,0.757,0.358,0.9914
3,Thinkin About,ShockOne;Lee Mvtthews,j-dance,Rock & Intenso,52,0.555,0.681,0.337,0.9881
4,Thinkin About,ShockOne;Lee Mvtthews,drum-and-bass,Rock & Intenso,52,0.555,0.681,0.337,0.9881
5,Fool Yourself,Chase & Status;Plan B;Rage,drum-and-bass,Rock & Intenso,22,0.471,0.781,0.255,0.9872
6,30/90,Andrew Garfield;Joshua Henry;Vanessa Hudgens;R...,show-tunes,Rock & Intenso,67,0.466,0.789,0.360,0.9867
7,BODY,LICK;LUNA AURA,club,Rock & Intenso,43,0.488,0.674,0.370,0.9861
8,Danger Line,Avenged Sevenfold,metal,Rock & Intenso,58,0.473,0.767,0.375,0.9854
9,Broken,Netsky;Montell2099,drum-and-bass,Rock & Intenso,54,0.518,0.806,0.327,0.9849
10,Blinding Lights,The Weeknd,pop,Rock & Intenso,3,0.512,0.796,0.344,0.9820


Comprobamos que muestran valores muy similares entre entre los parámetros, y que pertenecen todos al mismo cluster. 


In [6]:
# Segunda prueba.
titulo = "Shape of You"
artista = "Ed Sheeran"
n = 10
recomendaciones = recomendar_canciones(titulo, artista, n)
recomendaciones

Canción de referencia:
   Shape of You — Ed Sheeran
    Género: pop | Cluster: Fiesta & Baile
    Popularidad: 86


,track_name,artists,track_genre,cluster_nombre,popularity,danceability,energy,valence,similitud
1,Tene,Larry Gaaga;Flavour,dancehall,Fiesta & Baile,0,0.830,0.766,0.932,0.9768
2,Yaaro,Santesh;Amos Paul,malay,Fiesta & Baile,29,0.801,0.673,0.789,0.9766
3,Brujeria,El Gran Combo De Puerto Rico,salsa,Fiesta & Baile,65,0.795,0.623,0.960,0.9766
4,Go-Go Club - Raw,Vybz Kartel,j-dance,Fiesta & Baile,20,0.786,0.767,0.863,0.9758
5,Pop the Bubbles,Patty Shukla,kids,Fiesta & Baile,33,0.820,0.721,0.904,0.9749
6,Separemos Nuestras Vidas,Jerry Rivera,salsa,Fiesta & Baile,30,0.803,0.702,0.894,0.9739
7,Quiero Llenarte,Jerry Rivera,salsa,Fiesta & Baile,35,0.814,0.716,0.841,0.9723
8,Low,Larry Gaaga;Wizkid,dancehall,Fiesta & Baile,54,0.762,0.587,0.772,0.9716
9,Pull Up,Timaya;Burna Boy,dancehall,Fiesta & Baile,53,0.847,0.700,0.880,0.9706
10,Whine It,Jahyanai;Timal,dancehall,Fiesta & Baile,36,0.723,0.646,0.799,0.9701


El da unos resultados muy buenos basándose en el clúster y las similitudes. Lo que pasa, que la función de proximidad de coseno no filtra por contextos o géneros, es decir, un grupo de rock y otro de pop pueden tener las mismas características en una canción, y no ser de los mismos estilos.

Por tanto, la siguiente función que se implementará será la de filtrado por cluster y género.

# 4. Función de Recomendación v2.

Utilizando clúster y género.

In [9]:
def recomendar_por_genero_cluster(titulo, artista=None, n=10, mismo_genero=True,
                                   mismo_cluster=True, df=df, X_scaled=X_scaled):
    # Buscar canción de referencia
    mask = df['track_name'].str.contains(titulo, case=False, na=False)
    if artista:
        mask &= df['artists'].str.contains(artista, case=False, na=False)

    candidatos = df[mask]
    if candidatos.empty:
        print(f"❌ No se encontró ninguna canción con '{titulo}'")
        return None

    idx = candidatos['popularity'].idxmax()
    cancion_ref = df.loc[idx]

    print(f"🎵 Canción de referencia:")
    print(f"   {cancion_ref['track_name']} — {cancion_ref['artists']}")
    print(f"   Género: {cancion_ref['track_genre']} | Cluster: {cancion_ref['cluster_nombre']}")
    print(f"   Popularidad: {cancion_ref['popularity']}")
    print()

    # Filtrar espacio de búsqueda
    filtro = pd.Series([True] * len(df), index=df.index)
    if mismo_genero:
        filtro &= df['track_genre'] == cancion_ref['track_genre']
    if mismo_cluster:
        filtro &= df['cluster'] == cancion_ref['cluster']
    filtro.iloc[idx] = False

    indices_filtrados = df[filtro].index.tolist()

    if len(indices_filtrados) == 0:
        print("⚠️ No hay canciones suficientes. Prueba con mismo_genero=False.")
        return None

    print(f"🔍 Espacio de búsqueda: {len(indices_filtrados):,} canciones")

    # Similitud coseno
    vector_ref = X_scaled[idx].reshape(1, -1)
    X_filtrado = X_scaled[indices_filtrados]
    similitudes = cosine_similarity(vector_ref, X_filtrado)[0]

    # Score combinado
    sim_norm = (similitudes - similitudes.min()) / (similitudes.max() - similitudes.min() + 1e-9)
    pop_values = df.loc[indices_filtrados, 'popularity'].values
    pop_norm = (pop_values - pop_values.min()) / (pop_values.max() - pop_values.min() + 1e-9)
    score_final = 0.7 * sim_norm + 0.3 * pop_norm

    # Tomar más candidatos para compensar duplicados que se eliminarán
    n_candidatos = n * 5
    top_indices_locales = np.argsort(score_final)[::-1][:n_candidatos]
    top_indices_globales = [indices_filtrados[i] for i in top_indices_locales]

    recomendaciones = df.iloc[top_indices_globales][
        ['track_name', 'artists', 'track_genre', 'cluster_nombre',
         'popularity', 'danceability', 'energy', 'valence']
    ].copy()
    recomendaciones['similitud'] = similitudes[top_indices_locales].round(4)
    recomendaciones['score'] = score_final[top_indices_locales].round(4)

    # Eliminar títulos duplicados, quedarse con el de mayor score
    recomendaciones = (recomendaciones
                       .drop_duplicates(subset='track_name', keep='first')
                       .head(n)
                       .reset_index(drop=True))
    recomendaciones.index += 1

    return recomendaciones

In [10]:
# Prueba del nuevo recomendador.
recomendar_por_genero_cluster("Blinding Lights", artista="The Weeknd", n=10)

🎵 Canción de referencia:
   Blinding Lights — The Weeknd
   Género: pop | Cluster: Rock & Intenso
   Popularidad: 91

🔍 Espacio de búsqueda: 162 canciones


,track_name,artists,track_genre,cluster_nombre,popularity,danceability,energy,valence,similitud,score
1,STAY (with Justin Bieber),The Kid LAROI;Justin Bieber,pop,Rock & Intenso,89,0.591,0.764,0.4780,0.9420,0.9448
2,Unstoppable,Sia,pop,Rock & Intenso,81,0.468,0.779,0.2600,0.9657,0.9340
3,MIDDLE OF THE NIGHT,Elley Duhé,pop,Rock & Intenso,90,0.410,0.611,0.0899,0.8955,0.9220
4,Mann Mera,Gajendra Verma,pop,Rock & Intenso,74,0.535,0.765,0.3730,0.9565,0.9078
5,Wildest Dreams,Taylor Swift,pop,Rock & Intenso,80,0.553,0.664,0.4670,0.8930,0.8906
6,Shinunoga E-Wa,Fujii Kaze,pop,Rock & Intenso,85,0.600,0.760,0.5190,0.8489,0.8811
7,Choo Lo,The Local Train,pop,Rock & Intenso,67,0.512,0.695,0.3510,0.9286,0.8713
8,Under The Influence (Body Language),Chris Brown,pop,Rock & Intenso,77,0.620,0.620,0.4590,0.8355,0.8497
9,"Humraah (From ""Malang - Unleash The Madness"")",Sachet Tandon;The Fusion Project,pop,Rock & Intenso,67,0.468,0.854,0.2770,0.8853,0.8473
10,"Love Me Like You Do - From ""Fifty Shades Of Grey""",Ellie Goulding,pop,Rock & Intenso,79,0.262,0.606,0.2750,0.8052,0.8389


Estas canciones tienen mucho más sentido. La similitud es un un poco más baja que en la otra función, pero las canciones recomendadas son mucho más similares en cuanto a contextos musicales.

In [11]:
# Segunda prueba.
recomendar_por_genero_cluster("Shape of You", artista="Ed Sheeran", n=10)

🎵 Canción de referencia:
   Shape of You — Ed Sheeran
   Género: pop | Cluster: Fiesta & Baile
   Popularidad: 86

🔍 Espacio de búsqueda: 506 canciones


,track_name,artists,track_genre,cluster_nombre,popularity,danceability,energy,valence,similitud,score
1,Left and Right (Feat. Jung Kook of BTS),Charlie Puth;Jung Kook;BTS,pop,Fiesta & Baile,92,0.881,0.592,0.719,0.9265,0.9714
2,Calm Down (with Selena Gomez),Rema;Selena Gomez,pop,Fiesta & Baile,92,0.801,0.806,0.802,0.9173,0.9654
3,There's Nothing Holdin' Me Back,Shawn Mendes,pop,Fiesta & Baile,86,0.866,0.813,0.969,0.9003,0.9355
4,"Srivalli (From ""Pushpa The Rise Part - 01"")",Javed Ali;Devi Sri Prasad,pop,Fiesta & Baile,74,0.787,0.603,0.743,0.9511,0.9312
5,Closer,The Chainsmokers;Halsey,pop,Fiesta & Baile,84,0.748,0.524,0.661,0.8826,0.9177
6,"Mayakkama Kalakkama (From ""Thiruchitrambalam"")",Dhanush;Anirudh Ravichander,pop,Fiesta & Baile,77,0.789,0.536,0.877,0.9072,0.9119
7,Woman,Doja Cat,pop,Fiesta & Baile,88,0.824,0.764,0.881,0.8298,0.8956
8,Kya Baat Ay,Harrdy Sandhu,pop,Fiesta & Baile,68,0.849,0.592,0.652,0.9139,0.8882
9,"Naah Goriye (From ""Bala"")",B Praak;Harrdy Sandhu;Swasti Mehul,pop,Fiesta & Baile,64,0.823,0.765,0.715,0.9277,0.8847
10,Attention,Charlie Puth,pop,Fiesta & Baile,83,0.775,0.613,0.797,0.8335,0.8824


In [19]:
# Tercera prueba.
recomendar_por_genero_cluster("Californication", artista="Red Hot Chili Peppers", n=10)

🎵 Canción de referencia:
   Californication — Red Hot Chili Peppers
   Género: alt-rock | Cluster: Rock & Intenso
   Popularidad: 82

🔍 Espacio de búsqueda: 421 canciones


,track_name,artists,track_genre,cluster_nombre,popularity,danceability,energy,valence,similitud,score
1,Sweater Weather,The Neighbourhood,alt-rock,Rock & Intenso,93,0.612,0.807,0.3980,0.8705,0.9371
2,Cry Baby,The Neighbourhood,alt-rock,Rock & Intenso,74,0.581,0.656,0.3380,0.9412,0.9102
3,A Little Death,The Neighbourhood,alt-rock,Rock & Intenso,65,0.530,0.763,0.3380,0.9544,0.8876
4,You Get Me So High,The Neighbourhood,alt-rock,Rock & Intenso,83,0.551,0.881,0.3870,0.8308,0.8856
5,The Reason,Hoobastank,alt-rock,Rock & Intenso,81,0.472,0.671,0.0681,0.8304,0.8789
6,Tongue Tied,Grouplove,alt-rock,Rock & Intenso,79,0.560,0.936,0.3710,0.8373,0.8758
7,This Is Amazing Grace,Phil Wickham,alt-rock,Rock & Intenso,65,0.512,0.814,0.2770,0.9255,0.8735
8,Softcore,The Neighbourhood,alt-rock,Rock & Intenso,86,0.575,0.568,0.3700,0.7716,0.8665
9,Our God,Chris Tomlin,alt-rock,Rock & Intenso,63,0.509,0.778,0.2160,0.9216,0.8652
10,Twisted,MISSIO,alt-rock,Rock & Intenso,66,0.577,0.789,0.1490,0.8985,0.8636


In [20]:
# Cuarta prueba.
recomendar_por_genero_cluster("Si Veo A Tu Mamá", artista="Bad Bunny", n=10)

🎵 Canción de referencia:
   Si Veo a Tu Mamá — Bad Bunny
   Género: latino | Cluster: Fiesta & Baile
   Popularidad: 82

🔍 Espacio de búsqueda: 864 canciones


,track_name,artists,track_genre,cluster_nombre,popularity,danceability,energy,valence,similitud,score
1,La Bachata,Manuel Turizo,latino,Fiesta & Baile,98,0.835,0.679,0.850,0.9339,0.9858
2,Yo No Soy Celoso,Bad Bunny,latino,Fiesta & Baile,85,0.872,0.588,0.930,0.9579,0.9602
3,Un Coco,Bad Bunny,latino,Fiesta & Baile,87,0.839,0.690,0.744,0.8943,0.9286
4,Tigini (Remix),Rvfv;Kikimoteleba,latino,Fiesta & Baile,76,0.782,0.655,0.662,0.9424,0.9234
5,Junio,Maluma,latino,Fiesta & Baile,81,0.850,0.756,0.896,0.8822,0.9030
6,Berlin,Zion & Lennox;Maria Becerra,latino,Fiesta & Baile,81,0.796,0.766,0.846,0.8821,0.9030
7,Feliz,Chimbala,latino,Fiesta & Baile,77,0.848,0.852,0.956,0.8862,0.8931
8,La Copa,Ozuna,latino,Fiesta & Baile,72,0.913,0.803,0.947,0.9056,0.8894
9,Aguacero,Bad Bunny,latino,Fiesta & Baile,84,0.861,0.645,0.668,0.8392,0.8867
10,Gasolina,Daddy Yankee,latino,Fiesta & Baile,82,0.852,0.797,0.741,0.8182,0.8681


Comprobamos que los resultados son correctos con distintos géneros. Realizaremos unas últimas pruebas y finalizaremos este notebook.

## Pruebas

In [21]:
# Probar con diferentes géneros para validar robustez.
canciones_prueba = [
    ("Bohemian Rhapsody", "Queen"),
    ("Bad Guy", "Billie Eilish"),
    ("Despacito", "Luis Fonsi"),
]

for titulo, artista in canciones_prueba:
    print("\n" + "=" * 55)
    resultado = recomendar_por_genero_cluster(titulo, artista=artista, n=5)
    if resultado is not None:
        print(resultado[['track_name', 'artists', 'popularity', 'similitud', 'score']].to_string())


🎵 Canción de referencia:
   Bohemian Rhapsody - Remastered 2011 — Queen
   Género: rock | Cluster: Acústico & Tranquilo
   Popularidad: 82

🔍 Espacio de búsqueda: 194 canciones
                      track_name         artists  popularity  similitud   score
1               Do I Wanna Know?  Arctic Monkeys          88     0.8450  0.9723
2                   No Surprises       Radiohead          82     0.8726  0.9672
3               I Wanna Be Yours  Arctic Monkeys          92     0.7961  0.9598
4                         Homage  Mild High Club          78     0.8729  0.9543
5  Stairway to Heaven - Remaster    Led Zeppelin          79     0.8379  0.9392

🎵 Canción de referencia:
   bad guy — Billie Eilish
   Género: electro | Cluster: Fiesta & Baile
   Popularidad: 84

🔍 Espacio de búsqueda: 500 canciones
                      track_name                      artists  popularity  similitud   score
1   bad guy (with Justin Bieber)  Billie Eilish;Justin Bieber          68     0.9384  0.9016
2

# 5. Exportación del modelo

In [22]:
import pickle
import os

os.makedirs('../models', exist_ok=True)

with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('../models/X_scaled.pkl', 'wb') as f:
    pickle.dump(X_scaled, f)

print("Scaler y matriz de features guardados en /models ✅")
print("Dataset de referencia: data/processed/tracks_clustered.csv")

Scaler y matriz de features guardados en /models ✅
Dataset de referencia: data/processed/tracks_clustered.csv


# 6. Resumen del modelo

In [25]:
print("-" * 55)
print("    RESUMEN MODELO — SPOTIFY RECOMMENDER")
print("-" * 55)

print(f"\n  CONFIGURACIÓN")
print(f"  Algoritmo:         Similitud Coseno")
print(f"  Features:          {len(variables_modelo)}")
print(f"  Escalado:          StandardScaler")
print(f"  Filtros:           Género + Cluster")
print(f"  Score:             70% similitud + 30% popularidad")

print(f"\n COBERTURA")
print(f"  Canciones:         {df.shape[0]:,}")
print(f"  Géneros:           {df['track_genre'].nunique()}")
print(f"  Clusters:          7")

print(f"\n DECISIONES DE DISEÑO")
print(f"  - Filtro por género evita recomendaciones incoherentes")
print(f"  - Filtro por cluster añade coherencia de audio")
print(f"  - Deduplicación por título evita versiones repetidas")
print(f"  - Popularidad como criterio de desempate")
print("-" * 55)

-------------------------------------------------------
    RESUMEN MODELO — SPOTIFY RECOMMENDER
-------------------------------------------------------

  CONFIGURACIÓN
  Algoritmo:         Similitud Coseno
  Features:          9
  Escalado:          StandardScaler
  Filtros:           Género + Cluster
  Score:             70% similitud + 30% popularidad

 COBERTURA
  Canciones:         113,422
  Géneros:           114
  Clusters:          7

 DECISIONES DE DISEÑO
  - Filtro por género evita recomendaciones incoherentes
  - Filtro por cluster añade coherencia de audio
  - Deduplicación por título evita versiones repetidas
  - Popularidad como criterio de desempate
-------------------------------------------------------
